# Tạo Audio Kể Chuyện Kinh Thánh — GIỌNG CŨ STUDIO MASTERING (Ngắt Nghỉ Tự Nhiên + Giọng Ấm Phòng Thu)

1. Bật **Thời gian chạy → Đổi loại thời gian chạy → T4 GPU**.
2. Tự động nạp đúng **giọng cũ (`ref_voice.wav`)** từ Google Drive.
3. **Nâng cấp giọng hay & ngắt nghỉ tự nhiên:**
   - Cắt câu chuẩn dấu câu `. ? !` (không cắt giữa câu), nghỉ linh hoạt theo dấu câu (`,` nghỉ ngắn `0.18s`, `.` nghỉ `PAUSE_SEC`, `? / !` nghỉ sâu hơn `+0.18s`, `[pause]` nghỉ `0.80s`).
   - Tự động đọc chậm ấm hơn `7%` tại các câu gắn tag `[slowly]` hoặc `[softly]`.
   - **`STUDIO_MASTERING`:** Lọc ấm giọng phòng thu (tăng ấm `180Hz`, rõ chữ `3.2kHz`, giảm xì `6.8kHz`, nén đều âm lượng + chuẩn YouTube `-16 LUFS`).
4. Bấm **▶** → chọn `*_script.md` *(tích `PREVIEW_MODE` để nghe thử nhanh 3 đoạn đầu)*.

In [ ]:
# @title CHỌN FILE KỊCH BẢN (.md / .txt) ĐỂ XUẤT MP3 TỰ ĐỘNG
SPEED = 0.95  # @param {type:"number"}
NFE_STEP = 24  # @param [16, 24, 32] {type:"raw"}
PAUSE_SEC = 0.35  # @param {type:"slider", min:0.15, max:0.8, step:0.05}
STUDIO_MASTERING = True  # @param {type:"boolean"}
PREVIEW_MODE = False  # @param {type:"boolean"}
PREVIEW_CHUNKS = 3  # @param {type:"slider", min:1, max:10, step:1}

import os
import re
import sys
import shutil
import importlib.util
import subprocess
from pathlib import Path
from importlib.resources import files as pkg_files
from IPython.display import Audio, display
from google.colab import drive, files

# 1. Kết nối Google Drive
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')

CACHE_DIR = Path('/content/drive/MyDrive/Bible_TTS_Cache')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
PIP_CACHE = CACHE_DIR / 'pip_cache'
PIP_CACHE.mkdir(parents=True, exist_ok=True)
REPO_DIR = Path('/content/F5-TTS-Vietnamese')

# 2. Tải mã nguồn & cài nhanh (~15 giây, bỏ numpy<=1.26.4 gây biên dịch C 7 phút trên Python 3.13)
if not REPO_DIR.exists():
    print('>> Đang tải mã nguồn F5-TTS-Vietnamese...')
    subprocess.run(['git', 'clone', '-q', 'https://github.com/nguyenthienhy/F5-TTS-Vietnamese.git', str(REPO_DIR)], check=True)
    pyproject = REPO_DIR / 'pyproject.toml'
    txt = pyproject.read_text(encoding='utf-8')
    for drop in ['"numpy<=1.26.4",', '"gradio>=3.45.2",', '"transformers_stream_generator",', '"bitsandbytes>0.37.0; platform_machine != \'arm64\' and platform_system != \'Darwin\'",']:
        txt = txt.replace(drop, '')
    pyproject.write_text(txt, encoding='utf-8')

os.chdir(REPO_DIR)
if str(REPO_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_DIR / 'src'))

if importlib.util.find_spec('f5_tts') is None or shutil.which('f5-tts_infer-cli') is None:
    print('>> Đang cài gói F5-TTS (~15-20 giây)...')
    subprocess.run(['pip', 'install', '-q', '--cache-dir', str(PIP_CACHE), '-e', '.', 'omegaconf'], check=True)

import torch
import torchaudio
import numpy as np
import soundfile as sf
from omegaconf import OmegaConf
from tqdm.auto import tqdm as tqdm_cls
from f5_tts.model import DiT, UNetT
from f5_tts.infer.utils_infer import (
    device as f5_device,
    load_vocoder,
    load_model,
    preprocess_ref_audio_text,
    chunk_text,
    infer_batch_process,
    target_rms,
    cross_fade_duration,
    cfg_strength,
    sway_sampling_coef,
)

if not torch.cuda.is_available():
    print('⚠️ CẢNH BÁO: Đang chạy CPU! Hãy bật Thời gian chạy -> Đổi loại -> T4 GPU.')
else:
    print(f'✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}')

# 3. Nạp Model & Giọng mẫu từ Google Drive Cache
model_dir = REPO_DIR / 'F5-TTS-Vietnamese-ViVoice'
model_dir.mkdir(exist_ok=True)
ckpt_local = model_dir / 'model_last.pt'
vocab_local = model_dir / 'vocab.txt'
ckpt_drive = CACHE_DIR / 'model_last.pt'
vocab_drive = CACHE_DIR / 'vocab.txt'

if not ckpt_local.exists() or not vocab_local.exists():
    if ckpt_drive.exists() and vocab_drive.exists():
        print('>> Đang nạp Model từ Google Drive Cache...')
        shutil.copy(ckpt_drive, ckpt_local)
        shutil.copy(vocab_drive, vocab_local)
    else:
        print('>> Chưa có trong Drive — đang tải Model từ Hugging Face (1 lần)...')
        from huggingface_hub import hf_hub_download
        for fname in ['config.json', 'model_last.pt']:
            hf_hub_download(repo_id='hynt/F5-TTS-Vietnamese-ViVoice', filename=fname, local_dir=str(model_dir))
        shutil.copy(model_dir / 'config.json', vocab_local)
        shutil.copy(ckpt_local, ckpt_drive)
        shutil.copy(vocab_local, vocab_drive)

# Dùng lại đúng giọng cũ (ref_voice.wav) đang lưu sẵn trong Google Drive Cache
ref_local = REPO_DIR / 'ref_voice.wav'
ref_drive = CACHE_DIR / 'ref_voice.wav'
if not ref_local.exists():
    if ref_drive.exists():
        print('✅ Đang nạp lại giọng cũ (ref_voice.wav) từ Google Drive...')
        shutil.copy(ref_drive, ref_local)
    else:
        print('>> Chưa thấy ref_voice.wav trên Drive — vui lòng chọn file giọng cũ (.wav)...')
        up_ref = files.upload()
        ref_name = next(iter(up_ref.keys()))
        shutil.move(ref_name, ref_local)
        shutil.copy(ref_local, ref_drive)

REF_TEXT = 'hãy thử tưởng tượng về một người đàn ông sở hữu mọi thứ mà cả nhân loại suốt hàng ngàn năm qua vẫn ngày đêm khao khát.'

# Giữ sẵn Vocoder & Model trên RAM GPU suốt phiên Colab (từ lần thứ 2 không phải load lại)
cache = globals().get('_F5_CACHE')
if not cache or not Path(cache['ref_audio_proc']).exists() or cache.get('ref_text_raw') != REF_TEXT:
    print('>> Đang đưa Model & Vocoder lên GPU RAM (chỉ 1 lần trong phiên)...')
    vocoder = load_vocoder(vocoder_name='vocos', is_local=False, device=f5_device)
    model_cfg = OmegaConf.load(str(pkg_files('f5_tts').joinpath('configs/F5TTS_Base.yaml'))).model
    model_cls = {'DiT': DiT, 'UNetT': UNetT}[model_cfg.backbone]
    ema_model = load_model(
        model_cls,
        model_cfg.arch,
        str(ckpt_local),
        mel_spec_type='vocos',
        vocab_file=str(vocab_local),
        device=f5_device,
    )
    ref_audio_proc, ref_text_proc = preprocess_ref_audio_text(str(ref_local), REF_TEXT, device=f5_device)
    _F5_CACHE = {
        'vocoder': vocoder,
        'ema_model': ema_model,
        'ref_audio_proc': ref_audio_proc,
        'ref_text_proc': ref_text_proc,
        'ref_text_raw': REF_TEXT,
    }
    print('✅ Đã giữ sẵn Model trên GPU RAM!')
else:
    print('✅ Dùng lại Model đang nằm sẵn trên GPU RAM!')

vocoder = _F5_CACHE['vocoder']
ema_model = _F5_CACHE['ema_model']
ref_audio_proc = _F5_CACHE['ref_audio_proc']
ref_text_proc = _F5_CACHE['ref_text_proc']

def clean_script_text(raw: str) -> str:
    part_a = raw.split('## PHẦN B', 1)[0]
    kept = []
    for ln in part_a.splitlines():
        s = ln.strip()
        if not s or s.startswith('#') or set(s) <= {'-', '_'}: continue
        s = s.replace('**', '').replace('*', '').replace('`', '')
        kept.append(s)
    text = ' '.join(kept)
    # Giữ [pause] để nghỉ sâu, giữ [slowly]/[softly] để hạ tốc độ đọc ấm chậm hơn 7%
    text = re.sub(r'\[pause\]', ' ||PAUSE|| ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[(slowly|softly|whispering)\]', ' ||SLOW|| ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[[^\]]+\]\s*', '', text)
    text = text.replace('—', ', ').replace(':', ', ').replace(';', ', ')
    text = text.replace('"', '').replace('“', '').replace('”', '')
    text = re.sub(r'\s+', ' ', text).strip().lower()
    # Chuẩn hóa phát âm tên Kinh Thánh Công giáo & bỏ gạch nối cho F5-TTS đọc mượt
    tts_map = {
        'giê-su': 'giê xu',
        'giu-se': 'giu xe',
        'mô-sê': 'mô xê',
        'giô-suê': 'giô xuê',
        'giê-ru-sa-lem': 'giê ru xa lem',
        'sa-lô-môn': 'xa lô môn',
        'sa-un': 'xa un',
        'sa-mu-en': 'xa mu en',
        'bát-se-va': 'bát xe va',
        'bơ-e se-va': 'bơ e xe va',
        'sơ-va': 'xơ va',
        'cê-ci-li-a': 'xê xi li a',
    }
    for k, v in tts_map.items():
        text = text.replace(k, v)
    text = re.sub(r'(?<=[a-zà-ỹđ])-(?=[a-zà-ỹđ])', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def smart_chunk_vi(text: str, target_chars: int = 150, max_chars: int = 215):
    """Cắt câu chuẩn dấu câu (. ? !), nghỉ linh hoạt theo dấu câu và điều chỉnh tốc độ tại [slowly]/[softly].
    Trả về list[(chunk_text, pause_after_sec, speed_mult)]."""
    blocks = [b.strip() for b in text.split('||pause||') if b.strip()]
    chunks = []
    for b_idx, block in enumerate(blocks):
        sents = [s.strip() for s in re.split(r'(?<=[.?!])\s+', block) if s.strip()]
        norm_sents = []
        for s in sents:
            is_slow = '||slow||' in s
            s_clean = s.replace('||slow||', '').strip()
            s_clean = re.sub(r'\s+', ' ', s_clean)
            if not s_clean:
                continue
            while len(s_clean) > max_chars and ',' in s_clean:
                mid = len(s_clean) // 2
                commas = [m.start() for m in re.finditer(r',', s_clean)]
                split_at = min(commas, key=lambda pos: abs(pos - mid))
                left = s_clean[:split_at + 1].strip()
                right = s_clean[split_at + 1:].strip()
                if len(left) < 40 or len(right) < 40:
                    break
                norm_sents.append((left, is_slow))
                s_clean = right
            norm_sents.append((s_clean, is_slow))

        cur_txt = ''
        cur_slow = False
        for s_txt, s_slow in norm_sents:
            # Không gộp câu thường với câu [slowly]/[softly] để giữ sắc thái riêng
            if not cur_txt:
                cur_txt, cur_slow = s_txt, s_slow
            elif (not cur_slow and not s_slow) and (len(cur_txt) + 1 + len(s_txt) <= target_chars) and not cur_txt.endswith(('?', '!')):
                cur_txt = cur_txt + ' ' + s_txt
            else:
                chunks.append((cur_txt, cur_slow, False))
                cur_txt, cur_slow = s_txt, s_slow
        if cur_txt:
            has_deep_pause = (b_idx < len(blocks) - 1)
            chunks.append((cur_txt, cur_slow, has_deep_pause))

    result = []
    base_p = float(PAUSE_SEC)
    for seg_txt, is_slow, has_deep_pause in chunks:
        if has_deep_pause:
            pause_sec = max(0.80, base_p * 2.1)
        elif seg_txt.endswith('?'):
            pause_sec = base_p + 0.18
        elif seg_txt.endswith('!'):
            pause_sec = base_p + 0.15
        elif seg_txt.endswith(','):
            pause_sec = min(0.18, base_p * 0.55)
        else:
            pause_sec = base_p
        speed_mult = 0.93 if is_slow else 1.0
        result.append((seg_txt, round(pause_sec, 2), speed_mult))
    return result
class _ColabProgress:
    @staticmethod
    def tqdm(iterable, **kwargs):
        return tqdm_cls(iterable, unit='đoạn', **kwargs)

# 4. Chọn 1 hoặc nhiều file kịch bản (.md / .txt) từ máy tính
print('>> Bấm Choose Files bên dưới để chọn file kịch bản (.md hoặc .txt)...')
uploaded_scripts = files.upload()

ref_tensor, ref_sr = torchaudio.load(ref_audio_proc)
ref_sec = ref_tensor.shape[-1] / ref_sr
max_chars = max(80, int(len(ref_text_proc.encode('utf-8')) / ref_sec * (22 - ref_sec)))

for fname in uploaded_scripts.keys():
    src_path = Path(fname)
    clean_txt = clean_script_text(src_path.read_text(encoding='utf-8'))
    src_path.unlink(missing_ok=True)
    if not clean_txt:
        print(f'⚠️ Bỏ qua {fname} vì không có nội dung sau khi làm sạch.')
        continue

    stem = Path(fname).stem.replace('_script_clean_voice', '').replace('_script', '')
    chunk_items = smart_chunk_vi(clean_txt, target_chars=min(150, max_chars), max_chars=min(215, max_chars + 35))
    total_batches = len(chunk_items)
    if PREVIEW_MODE:
        chunk_items = chunk_items[:int(PREVIEW_CHUNKS)]
        out_mp3 = f'{stem}_preview.mp3'
        preview_words = sum(len(b[0].split()) for b in chunk_items)
        print(f'\n>> [PREVIEW MODE] Đang đọc {len(chunk_items)}/{total_batches} đoạn đầu ({preview_words} từ) của {fname} | SPEED={SPEED} | NFE_STEP={int(NFE_STEP)} | PAUSE={PAUSE_SEC}s')
    else:
        out_mp3 = f'{stem}.mp3'
        word_count = len(clean_txt.replace('||pause||', '').replace('||slow||', '').split())
        est_audio_min = word_count / (195.0 * float(SPEED))
        print(f'\n>> Đang đọc FULL {fname}: {word_count} từ | {total_batches} đoạn | SPEED={SPEED} | NFE_STEP={int(NFE_STEP)} | PAUSE={PAUSE_SEC}s')
        print(f'   ⏱️ Thời lượng audio ước tính: ~{est_audio_min:.1f} phút (xem thanh % và ETA bên dưới)')

    # Render từng cụm câu chuẩn dấu câu và chèn khoảng nghỉ mềm chống tiếng click
    wave_parts = []
    final_sr = 24000
    for seg_txt, pause_after, speed_mult in tqdm_cls(chunk_items, unit='đoạn'):
        seg_wave, final_sr, _ = next(
            infer_batch_process(
                (ref_tensor, ref_sr),
                ref_text_proc,
                [seg_txt],
                ema_model,
                vocoder,
                mel_spec_type='vocos',
                progress=None,
                target_rms=target_rms,
                cross_fade_duration=0.0,
                nfe_step=int(NFE_STEP),
                cfg_strength=cfg_strength,
                sway_sampling_coef=sway_sampling_coef,
                speed=float(SPEED) * speed_mult,
                fix_duration=None,
                device=f5_device,
            )
        )
        seg_wave = np.asarray(seg_wave, dtype=np.float32)
        fade_n = min(int(final_sr * 0.015), len(seg_wave) // 4)
        if fade_n > 0:
            seg_wave[:fade_n] *= np.linspace(0.0, 1.0, fade_n, dtype=np.float32)
            seg_wave[-fade_n:] *= np.linspace(1.0, 0.0, fade_n, dtype=np.float32)
        wave_parts.append(seg_wave)
        wave_parts.append(np.zeros(int(final_sr * pause_after), dtype=np.float32))
    final_wave = np.concatenate(wave_parts) if wave_parts else np.zeros(1, dtype=np.float32)

    out_audio_dir = REPO_DIR / 'out_audio'
    shutil.rmtree(out_audio_dir, ignore_errors=True)
    out_audio_dir.mkdir(exist_ok=True)
    raw_wav = out_audio_dir / f'{stem}.wav'
    sf.write(str(raw_wav), final_wave, final_sr)

    af_chain = (
        'highpass=f=75,'
        'equalizer=f=180:t=q:w=1.2:g=2.5,'
        'equalizer=f=3200:t=q:w=1.5:g=1.5,'
        'equalizer=f=6800:t=q:w=2.0:g=-2.5,'
        'acompressor=threshold=-20dB:ratio=2.5:attack=15:release=180:makeup=2,'
        'loudnorm=I=-16:TP=-1.5:LRA=9'
    ) if STUDIO_MASTERING else 'loudnorm=I=-16:TP=-1.5:LRA=11'
    subprocess.run([
        'ffmpeg', '-y', '-loglevel', 'warning', '-i', str(raw_wav),
        '-af', af_chain,
        '-ar', '44100', '-ac', '2', '-b:a', '192k', out_mp3
    ], check=True)

    shutil.copy(out_mp3, CACHE_DIR / out_mp3)
    if PREVIEW_MODE:
        print(f'🎧 Nghe thử {out_mp3} ngay bên dưới (bỏ tích PREVIEW_MODE để xuất bản full):')
        display(Audio(out_mp3))
    else:
        print(f'✅ Hoàn tất {out_mp3}! Đã lưu Drive và đang tải về máy...')
        files.download(out_mp3)